In [1]:
import pandas as pd
import plotly.express as px
from pathlib import Path

RAW = Path("../data/raw")
file = next(RAW.glob("*berufe-heft-kldb2010*202610*.xlsx"))

rows = []
for i in range(4, 20):                       # листы 1.4 … 1.19 = 16 федеральных земель
    sheet = f"1.{i}"
    region = pd.read_excel(file, sheet_name=sheet, header=None, skiprows=5, nrows=1).iloc[0, 0]
    t = pd.read_excel(file, sheet_name=sheet, header=None, skiprows=15, nrows=5).iloc[:, [2, 3, 11]]
    t.columns = ["level", "unemployed", "vacancies"]
    t["level"] = t["level"].str.strip()
    t["region"] = region[3:]                 # "06 Hessen" → "Hessen"
    rows.append(t)

reg = pd.concat(rows)
reg["unemployed_per_vacancy"] = (reg["unemployed"] / reg["vacancies"]).round(1)
table = reg.pivot(index="region", columns="level", values="unemployed_per_vacancy")
table = table[["Gesamt", "Helfer", "Fachkraft", "Spezialist", "Experte"]].sort_values("Gesamt")
table

level,Gesamt,Helfer,Fachkraft,Spezialist,Experte
region,,,,,
Bayern,3.0,7.7,1.4,2.6,3.7
Brandenburg,3.9,10.7,1.9,2.6,2.9
Rheinland-Pfalz,3.9,10.0,1.8,2.1,3.1
Baden-Württemberg,4.0,9.7,2.0,2.9,3.8
Mecklenburg-Vorpommern,4.0,13.3,2.0,1.7,1.8
Schleswig-Holstein,4.1,11.9,1.9,2.2,3.5
Thüringen,4.2,10.8,2.2,2.1,2.7
Niedersachsen,4.4,12.2,2.0,2.3,3.4
Sachsen,4.8,16.7,2.3,2.9,3.6


In [2]:
fig = px.imshow(table, text_auto=True, aspect="auto", color_continuous_scale="Reds", height=650,
                title="Unemployed per vacancy by state and skill level, October 2026",
                labels={"x": "Skill level", "y": "", "color": "Unemployed per vacancy"})
fig.show()
fig.write_html("../output/charts/05_regions_heatmap.html")

## Finding 5: The paradox exists in every state

- In all 16 states, there are far more unemployed per vacancy at **Helfer** level (7.7 in Bavaria to 39.8 in Berlin) than at **Fachkraft** level (1.4 to 5.0).
- Bavaria has the tightest skilled labour market (1.4 unemployed per Fachkraft vacancy), Berlin the loosest.
- **Hessen** is in the middle (4.8 overall; Helfer 11.3, Fachkraft 2.3), but with a relatively high ratio for Experten (3.9).
- The mismatch is a nationwide structural problem, not a regional one. Its intensity varies strongly by region.

Source: BA, Arbeitsmarkt nach Berufen, October 2026. Own calculations.